# 24. Robustness check — another measure of the truth
**Part 5 — Checks, extensions and prediction.** All scores so far compare the forecasts with the realised variance from **closing prices**. Would the ranking of the models change with a different measure of the 'truth'? The **Parkinson** measure uses each day's **high and low** prices.

* **Needs:** `work/dataset.csv` and the three forecast files in `work/`. **Makes:** nothing new.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

In [2]:
from pandas import read_csv, DataFrame
dataset = read_csv(folder + 'work/dataset.csv', index_col=0, parse_dates=True)
har = read_csv(folder + 'work/har_forecasts.csv', index_col=0, parse_dates=True)
garch = read_csv(folder + 'work/garch_forecasts.csv', index_col=0, parse_dates=True)
lstm = read_csv(folder + 'work/lstm_forecasts.csv', index_col=0, parse_dates=True)

## The Parkinson measure
$\sigma^2_{day} = \dfrac{(\ln \text{High} - \ln \text{Low})^2}{4 \ln 2}$ — a day with a wide range between high and low was a volatile day. Summing the next 5 days gives a second 'truth' for next week.

In [3]:
from numpy import log
dataset['park'] = log(dataset['High'] / dataset['Low']) ** 2 / (4 * log(2))
dataset['park5'] = dataset['park'].shift(-1) + dataset['park'].shift(-2) + dataset['park'].shift(-3) + dataset['park'].shift(-4) + dataset['park'].shift(-5)
dataset[['rv5_fwd', 'park5']].loc['2008-10-01':'2008-10-08']

,rv5_fwd,park5
2008-10-01,0.025657,0.015608
2008-10-02,0.023317,0.015026
2008-10-03,0.025529,0.017530
2008-10-06,0.015242,0.014673
2008-10-07,0.010477,0.010715
2008-10-08,0.008878,0.009423


The high–low range misses the move overnight (between one close and the next open), so the Parkinson numbers are put on the same level as the close-to-close numbers, using **2003–2006 only** (before any test day):

In [4]:
scale = dataset.loc['2003':'2006', 'rv5_fwd'].mean() / dataset.loc['2003':'2006', 'park5'].mean()
print('scale factor =', round(scale, 3))

scale factor = 1.001


## Score the same forecasts against the new truth

In [5]:
results = DataFrame({'truth': scale * dataset.loc[har.index, 'park5'],
                     'LSTM': lstm['LSTM'], 'GARCH': garch['GARCH'], 'HAR': har['HAR'],
                     'window': dataset.loc[har.index, 'window']})
results = results[results['truth'] > 0]
for m in ['LSTM', 'GARCH', 'HAR']:
    ratio = results['truth'] / results[m]
    results['QLIKE_' + m] = ratio - log(ratio) - 1
results[['QLIKE_LSTM', 'QLIKE_GARCH', 'QLIKE_HAR']].mean().round(3)

QLIKE_LSTM     0.295
QLIKE_GARCH    0.237
QLIKE_HAR      0.270
dtype: float64

In [6]:
results.groupby('window')[['QLIKE_LSTM', 'QLIKE_GARCH', 'QLIKE_HAR']].mean().round(3)

,QLIKE_LSTM,QLIKE_GARCH,QLIKE_HAR
window,,,
Brexit referendum,2.368,1.883,1.872
COVID-19,0.315,0.509,0.525
Global Financial Crisis,0.638,0.256,0.416
Irish sovereign debt crisis,0.276,0.232,0.255
War / energy shock 2022,0.250,0.161,0.163
calm,0.224,0.208,0.229


In [7]:
from numpy import sqrt
from scipy.stats import t


def dm_test(loss_a, loss_b):
    '''The Diebold-Mariano test of notebook 13 (negative difference = model A more accurate).'''
    d = loss_a - loss_b
    n = len(d)
    d_mean = d.mean()
    dc = d - d_mean
    long_run_variance = (dc * dc).sum() / n
    for k in [1, 2, 3, 4]:
        long_run_variance = long_run_variance + 2 * (dc * dc.shift(k)).sum() / n
    dm = d_mean / sqrt(long_run_variance / n)
    dm_hln = dm * sqrt((n + 1 - 10 + 20 / n) / n)
    p_value = 2 * t.sf(abs(dm_hln), df=n - 1)
    print('average difference =', round(d_mean, 4), '| DM =', round(dm_hln, 2), '| p-value =', round(p_value, 4))
    return d_mean, p_value

In [8]:
print('LSTM vs GARCH:')
dm_test(results['QLIKE_LSTM'], results['QLIKE_GARCH'])
print('LSTM vs HAR:')
dm_test(results['QLIKE_LSTM'], results['QLIKE_HAR'])

LSTM vs GARCH:
average difference = 0.0574 | DM = 3.26 | p-value = 0.0011
LSTM vs HAR:
average difference = 0.0241 | DM = 1.5 | p-value = 0.1334


(np.float64(0.024069483013794097), np.float64(0.13337112430209314))

In [9]:
saved = read_csv(folder + 'results/robust_parkinson_losses.csv', index_col=0)
saved.filter(like='QLIKE').round(3)

,QLIKE LSTM,QLIKE GARCH,QLIKE HAR
window,,,
All test days (2007 - Aug 2023),0.295,0.237,0.270
Brexit referendum,2.368,1.883,1.872
COVID-19,0.315,0.509,0.525
Global Financial Crisis,0.638,0.256,0.416
Irish sovereign debt crisis,0.276,0.232,0.255
War / energy shock 2022,0.250,0.161,0.163
calm,0.224,0.208,0.229


**Result:** GARCH is still the most accurate overall, the LSTM is still significantly worse than GARCH (but no longer significantly worse than HAR), and the LSTM is still the best model in COVID-19. The main conclusions do not depend on how the 'truth' is measured.